# Part 3: Prompt Engineering Science
 
> **Central idea:** Prompting is engineering, not art. Controlled experiments reveal which components actually move the needle, and by how much.

In [ ]:
# imports

import json
import time
import re
import mlflow
from chat.client import ChatClient
from utils.tracker import setup_mlflow
import pandas as pd
from IPython.display import display
import uuid
import random

In [ ]:
def normalize(value):
    # Normalize the response text (e.g. remove trailing whitespace, etc)
    return value


def extract_json_object(text):
    """
    Extract the first valid JSON object from a model response.
    Prefers fenced JSON blocks when available.
    """

    return {}

In [ ]:
def f1_evaluator(predicted_records, ground_truth_records):
    """
    Calculates precision, recall, and F1 for exact-match field extraction.

    We treat the target as a set of key-value pairs over the schema:
    {"id" -> "sector"}.
    """
    
    precision = ...
    recall = ...
    f1_score = ...

    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
    }

In [ ]:
NUM_TARGETS = 100
MIDDLE_LO = 0.40
MIDDLE_HI = 0.60

def parse_record(record_text):
    parts = record_text.strip().split(" | ")
    return {
        "id": parts[0].split(": ", 1)[1].strip(),
        "clearance": parts[1].split(": ", 1)[1].strip(),
        "sector": parts[2].split(": ", 1)[1].strip(),
    }


def get_targets_from_middle(haystack, k=NUM_TARGETS, seed=42):
    """
    Select k target records from the middle 20% of the haystack.
    Returns full records so we can derive the gold id->sector mapping.
    """
    rng = random.Random(seed)
    mid_start = int(len(haystack) * MIDDLE_LO)
    mid_end = int(len(haystack) * MIDDLE_HI)

    candidates = haystack[mid_start:mid_end]
    if k > len(candidates):
        raise ValueError("k is larger than the number of middle candidates.")

    chosen = rng.sample(candidates, k)
    return [parse_record(record) for record in chosen]

In [ ]:
def build_user_input(haystack_text, target_records):
    target_ids = [record["id"] for record in target_records]
    return (
        "Read the haystack below and find the sectors for the target record ids listed below. "
        "Return as a JSON with each target Record-ID matching its corresponding Sector. "
        f"Target Record-IDs: {json.dumps(target_ids)}\n\n"
        f"Haystack:\n{haystack_text}"
    )

In [ ]:
num_records = 2000
haystack_lines = []

while len(haystack_lines) < num_records:
    random_id = uuid.uuid4().hex[:8]
    record = f"Record-ID: {random_id} | Clearance: {random.randint(1,5)} | Sector: {uuid.uuid4().hex[:6]}\n"
    haystack_lines.append(record)

test_paragraph = "\n".join(haystack_lines)

In [ ]:
target_records = get_targets_from_middle(haystack_lines, k=NUM_TARGETS, seed=42)
ground_truth_record = {record["id"]: record["sector"] for record in target_records}
user_input = build_user_input(test_paragraph, target_records)

## Task 3.1: PTCF Ablation Study

Instructions:
- Aim for an F1 score around 0.9

- Do NOT change the model

In [ ]:
client = ChatClient(budgeting=False, model="databricks-gemma-3-12b")
registry = client.prompt_registry

In [ ]:
baseline_prompt = (
    "**Input:** {{ user_input }}"
)

p_persona = (
    "**Persona:** \n\n"
)

p_task = (
    "**Task:** \n\n"
)

p_context = (
    "**Context:** \n\n"
)

p_format = (
    "**Format:** \n\n"
)

p_input = "**Input:** {{ user_input }}"

# Register all prompt variants
prompt_variants = {
    ...
}

# registry.create()
# registry.update()

In [ ]:
setup_mlflow("Task-3.1-PTCF-Ablation")

ablation_variants = [
    ...
]

ablation_results = []

In [ ]:
for variant in ablation_variants:
    print(f"Evaluating {variant}...")
    # start a run with MLflow

    ...

# Display a compact summary table

#### Analysis questions

1. Which single PTCF component had the largest marginal F1 gain when added to the simplest baseline? Report the exact scores before and after the change.
2. Did the full PTCF prompt always beat two-component prompts? Identify any cases where adding a component hurt performance and give a concrete explanation.
3. Did the model ever produce entity types outside the schema, such as person names or locations? Which prompt component reduced that behavior the most?
4. Your evaluator uses exact string matching. How would the scores change under fuzzy matching, and is exact-match a fair metric for this task? State the tradeoff clearly.

## Task 3.2: Prompting Strategies Comparison

In [ ]:
# Build the three strategy prompts from the best-performing PTCF variant found in Task 3.1
# (This cell assumes ablation_results has already been populated.)

ptcf_candidates = [row for row in ablation_results]
best_ptcf_variant = max(ptcf_candidates, key=lambda row: row["F1 Score"])["Variant"]
best_ptcf_template = prompt_variants[best_ptcf_variant]


In [ ]:
few_shot_examples = """

"""

chain_of_thought = """
"""

In [ ]:
strategy_zero_shot = ...
strategy_few_shot = ...
strategy_cot = ...

try:
    registry.create("strategy_zero_shot", strategy_zero_shot)
    ...
except ValueError:
    registry.update("strategy_zero_shot", strategy_zero_shot)
    ...

In [ ]:
setup_mlflow("Task-3.2-Prompt-Strategies")

strategies = [
    "strategy_zero_shot",
    "strategy_few_shot",
    "strategy_cot"
]

strategy_results = []

In [ ]:
for strategy in strategies:
    print(f"Evaluating {strategy}...")
    
    
# Display a compact summary table

#### Analysis questions

1. Rank the three strategies by F1 score. Does the ranking match your expectation? State any surprises.
2. Did CoT help, hurt, or make no measurable difference on this extraction task? Explain what that implies about when CoT is useful.
3. Few-shot examples consume context. At what point would adding more examples start to reduce performance because the document itself no longer fits comfortably? Explain how you would test that threshold.
4. You used 2-shot. Would 5-shot likely improve, stay flat, or degrade? Give a reasoned prediction and describe the experiment you would run to verify it.